# WINGS3 — EvalHub and Garak (Act 5)

Run in JupyterLab workbench **wings3-demo**, project `my-first-model`.

**Red thread step 5:** MLflow `v2-judged` proves *correctness* on golden rows. **EvalHub** operationalizes benchmarks as platform jobs. **Garak** stress-tests the same endpoint with adversarial prompts.

**Say this first:** Same agent endpoint (`MAAS_BASE_URL` from Secret `wings3-judge-llm`) the calculator agent called in Act 2. Judges can pass while the model still fails under attack.

On stage, stop at each **SHOW:** comment. Primary demo is **Develop & train → Evaluations** (RHOAI 3.5) — this notebook is the presenter aid.


## 0. Optional: git pull

Skip on stage if already current.


In [ ]:
# Optional: update from GitHub. Skip on stage if already current.
!git pull --ff-only


## 1. Read the shared endpoint

**SHOW:** This is the URL EvalHub and Garak must target — the same vLLM service Acts 2–4 used.


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

demo = Path("../agent-tracing").resolve()
sys.path.insert(0, str(demo))
traced_agent = importlib.import_module("traced_agent")
traced_agent.ensure_maas_env()

PROJECT = os.environ.get("MLFLOW_WORKSPACE", "my-first-model")
endpoint_url = os.environ["MAAS_BASE_URL"]
MODEL = os.environ["MAAS_MODEL"]
jobs_dir = Path("../evalhub/jobs")

print(f"Project: {PROJECT}")
print(f"Model:   {MODEL}")
print(f"Endpoint: {endpoint_url}")
print()
print("Job templates:", jobs_dir.resolve())


## 2. Evaluations console path (RHOAI 3.5)

**SHOW:** **Develop & train → Evaluations** → project `my-first-model` → **Start evaluation run**.

There is no project-level **EvalHub** sidebar tile on 3.5. Requires EvalHub CR `evalhub` in this namespace (`install.sh` applies `manifests/evalhub-instance.yaml`).

- Provider: **lm-eval-harness**
- Target: endpoint above
- Task: one small harness task (demo speed)
- Review metrics + pass/fail when complete


In [ ]:
lm_eval_job = json.loads((jobs_dir / "lm-eval-demo.json").read_text())
print(json.dumps(lm_eval_job, indent=2))


## 3. Garak red-team scan

**SHOW:** **Develop & train → Evaluations** → **Start evaluation run** → provider **Garak** → same endpoint.

When the job completes, open the **HTML report**. Walk one **resisted** and one **vulnerable** probe category. Do not read harmful model outputs aloud.


In [ ]:
garak_job = json.loads((jobs_dir / "garak-demo.json").read_text())
print(json.dumps(garak_job, indent=2))


## 4. Tie back to MLflow

**SHOW:** In `/mlflow` → experiment `wings3-agent-eval-prod` → Evaluation → run `v2-judged`.

> Judge said *correct*. Garak asks *safe under attack*. Full promotion gate: MLflow PASS → EvalHub PASS → Garak PASS → deploy + guardrails.

Walkthrough: `walkthrough/05-evalhub-garak.md`
